# 02. Optimization 
**About:** This notebook implements the **scenario-based Multi-Objective Linear Programming (MOLP)** model for hourly energy procurement. <br>

**Decision variables:** x1(t), x2(t) for each of 8760 hours - MWh procured from Standard Contract and RE-Certified Contract respectively. <br>
As x3(t) = D(t) - x1(t) - x2(t) follows as the residual spot market purchase, we just need to run the model for `x1`, `x2`

**Objective:**
$$ \min \quad w_1 \cdot \text{Cost} + w_2 \cdot\text{Supply Risk} + w_3 \cdot \text{CarbonIntensity} $$
where each term is min-max normalized using a **payoff table** derived from solving each objective independently.

**Sections:** <br>
1 Setup <br>
2 Model Parameters (market parameters, linear coefficients)<br>
3 Payoff Table (single-objective LPs)<br>
4 Weighted MOLP - 4 Profiles <br>

# 1. Setup & Load Data

In [61]:
#Set up and Load Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import linprog
from scipy.sparse import hstack, eye

ROOT_DIR = Path(__file__).parent.parent if '__file__' in dir() else Path().resolve().parent
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR       = ROOT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi':       120,
    'figure.facecolor': 'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
})

#LOAD DATA
df = pd.read_csv(PROCESSED_DIR / 'final_dataset.csv', parse_dates=['date'])
df = df.set_index('date').sort_index()
#Label scenario load_type x price_tier
df['scenario'] = df['load_type'] + ' / ' + df['price_tier'].astype(str)

n_hours = len(df)
print(f'Hours: {n_hours}')
print(f'Scenarios: {df["scenario"].nunique()}')
df[['demand_mwh', 'dayahead_price', 'balancing_price', 'ci_kgco2_mwh', 'scenario']].head(3)

Hours: 8760
Scenarios: 9


,demand_mwh,dayahead_price,balancing_price,ci_kgco2_mwh,scenario
date,,,,,
2018-01-01 00:00:00,7.205852,26.33,30.46999,107.207999,Light_Load / Cheap
2018-01-01 01:00:00,7.299637,26.43,30.00000,105.856999,Light_Load / Cheap
2018-01-01 02:00:00,7.356951,26.10,30.00000,117.173000,Light_Load / Cheap


# 2. Model Parameters

## 2.1. Market Parameters
Market parameters assumption: <br>
`p1`, `p2`: fixed contract price, assume 10%, 15% premium from average spot price, respectively. Should be higher than expected spot price as hedging over price volatility & GO certificates. <br>
`C1`, `C2`: capped quantity for each contract in each hour, assume 50% and 20%, respectively. <br>
`e`: fraction of failed delivery by each contract generator (assumed = 2%) <br>
`spread`: assume expected penalty from buying from balancing market is larger than 0 (as there exists hour that balancing price is lower)<br>
`carbon_price`: take average annual value of carbon price in 2018, 15.5 euro/tonnes

In [74]:
D = df['demand_mwh'].values
p3 = df['dayahead_price'].values
p_bal = df['balancing_price'].values
ci = df['ci_kgco2_mwh'].values
avg_D = D.mean()
E_spot = p3.mean()
#Contract price
p1 = E_spot * 1.10 
p2 = E_spot * 1.15 
#Contract capacity cap
C1 = 0.50 * avg_D 
C2 = 0.20 * avg_D 
e = 0.02 
spread = np.maximum(0, p_bal - p3)

# Carbon price - To make the CI compatible with cost & risk, we convert it into monetary value using Carbon price
carbon_price = 15.5 # Euro/tonnes
ci_eur = ci / 1e3 * carbon_price

print('Model Parameters:')
print(f'Average demand D(t):     {avg_D:.2f} MWh/hour')
print(f'Average spot price:      {E_spot:.2f} EUR/MWh')
print(f'p1 (standard contract):  {p1:.2f} EUR/MWh')
print(f'p2 (RE contract):        {p2:.2f} EUR/MWh')
print(f'C1_bar (standard cap):   {C1:.2f} MWh/hour  ({C1/avg_D*100:.0f}% of avg D)')
print(f'C2_bar (RE cap):         {C2:.2f} MWh/hour  ({C2/avg_D*100:.0f}% of avg D)')
print(f'pi (failure probability):{e}')
print(f'Carbone price:           {carbon_price} EUR/tonnes')

Model Parameters:
Average demand D(t):     57.08 MWh/hour
Average spot price:      46.80 EUR/MWh
p1 (standard contract):  51.48 EUR/MWh
p2 (RE contract):        53.82 EUR/MWh
C1_bar (standard cap):   28.54 MWh/hour  (50% of avg D)
C2_bar (RE cap):         11.42 MWh/hour  (20% of avg D)
pi (failure probability):0.02
Carbone price:           15.5 EUR/tonnes


## 2.2. Linear coefficient
`linprog` solves: $\min_x c^T.x$ subject to constraints, where $c^T$ is the coefficient at each `t` and $x$ is our decision variables - i.e. the allocation to each contract each hour. To solve, we need to determine the coefficient of each decision, at each hour. <br>
Basically, the coefficient tells how the whole objective changes if the decision at one period about one contract type changes. 

**Cost coefficient**: for each time `t`, $Cost(t) = x_1 . p_1 + x_2 . p_2 + (D(t)-x_1-x_2).p3 = x_1.(p_1-p_3) + x_2.(p_2-p_3) + D(t).p_3$ <br>
-> For this `t`, the coef for $x_1, x_2$ are `p1-p3`,`p2 - p3`, respectively.

**Supply risk**: $Risk (t) = \epsilon. (x_1 + x_2). (\text{balancing price}- \text{spot price})$ <br>
-> Coefs are:  `e.(p_bal - p3)`

**Carbon intensity**: $CI(t) = (x_1 + x_3)*ci$. <br>
-> Thus by reduced form, we exclude `x_3` (by the equation `x_3 = D-x_1-x_2`), then: $CI(t) = (D-x_3)*ci$ <br> 
-> Then the coef of `x_1`  is 0 and `x_2` is `- ci_eur`

In [41]:
T = n_hours
c_cost_x1 = (p1 - p3)
c_cost_x2 = (p2 - p3)
c_cost = np.concatenate([p1-p3,p2-p3])
c_risk = np.concatenate([e * spread, e* spread])
c_carbon = np.concatenate([np.zeros(T),-ci_eur])
print('Coefficient vectors built.')
print(f'  c_cost   shape: {c_cost.shape}')
print(f'  c_risk   shape: {c_risk.shape}')
print(f'  c_carbon shape: {c_carbon.shape}')

Coefficient vectors built.
  c_cost   shape: (17520,)
  c_risk   shape: (17520,)
  c_carbon shape: (17520,)


In [42]:
#The constant reduced reduced: 
cost_constant = (D*p3).sum()
risk_constant = 0.0
ci_constant = (D*ci_eur).sum()
print(f'\nConstants (added back to objective value for reporting):')
print(f'  Cost constant   (sum p3*D):  {cost_constant:,.2f} EUR')
print(f'  Carbon constant (sum ci*D):  {ci_constant:,.2f} EUR')


Constants (added back to objective value for reporting):
  Cost constant   (sum p3*D):  25,764,996.41 EUR
  Carbon constant (sum ci*D):  837,659.03 EUR


## 2.3. Constraints
`A_ub`: inequality matrix, specifies coef linear contraints on `x` <br> 
`b_ub`: upperbound for `A_ub` <br>
-> Constraint: `x1(t) + x2(t) <= D(t)` <br>
`bounds`: defining min/max value for `x`-> We cap `x1`, `x2` by `C1`, `C2`<br>

In [62]:
# Bounds: 0 <= x1(t) <= C1_bar,  0 <= x2(t) <= C2_bar
bounds = [(0, min(C1, D[t])) for t in range(T)] + [(0, min(C2, D[t])) for t in range(T)]

# A_ub x <= b_ub :  x1(t) + x2(t) <= D(t)
from scipy.sparse import hstack, eye
I_T = eye(T, format='csr')
A_ub = hstack([I_T, I_T], format='csr')
b_ub = D.copy()

print(f'A_ub shape: {A_ub.shape}')
print(f'b_ub shape: {b_ub.shape}')
print(f'Bounds: x1 in [0, {C1:.2f}], x2 in [0, {C2:.2f}]')

A_ub shape: (8760, 17520)
b_ub shape: (8760,)
Bounds: x1 in [0, 28.54], x2 in [0, 11.42]


# 3. Payoff Tables

## 3.1. Single-objective MOLP 
I adopt the single-objective MOLP approach, i.e. solve each sub-objective optimization problem individually and then take the weighted average. 

In [51]:
#define single-objective function
def evaluate_objectives(x):
    cost   = c_cost   @ x + cost_constant
    risk   = c_risk   @ x + risk_constant
    carbon = c_carbon @ x + ci_constant
    return cost, risk, carbon

def solve_lp(c):
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs-ipm',options={'time_limit': 15})
    if not res.success:
        raise RuntimeError(f'LP failed: {res.message}')
    return res.x

print('Solving single-objective LPs:')

x_min_cost = solve_lp(c_cost)
cost_at_minC, risk_at_minC, carbon_at_minC = evaluate_objectives(x_min_cost)
print(f'  min Cost   -> Cost={cost_at_minC:,.0f}  Risk={risk_at_minC:,.0f}  Carbon={carbon_at_minC:,.0f}')

x_min_risk = solve_lp(c_risk)
cost_at_minR, risk_at_minR, carbon_at_minR = evaluate_objectives(x_min_risk)
print(f'  min Risk   -> Cost={cost_at_minR:,.0f}  Risk={risk_at_minR:,.0f}  Carbon={carbon_at_minR:,.0f}')

x_min_carbon = solve_lp(c_carbon)
cost_at_minG, risk_at_minG, carbon_at_minG = evaluate_objectives(x_min_carbon)
print(f'  min Carbon -> Cost={cost_at_minG:,.0f}  Risk={risk_at_minG:,.0f}  Carbon={carbon_at_minG:,.0f}')

Solving single-objective LPs...
  min Cost   -> Cost=24,915,676  Risk=10,371  Carbon=811,573
  min Risk   -> Cost=25,764,996  Risk=0  Carbon=837,659
  min Carbon -> Cost=26,242,891  Risk=13,517  Carbon=691,803


In [75]:
payoff = pd.DataFrame({
    'Cost [EUR]': [cost_at_minC,   cost_at_minR,   cost_at_minG],
    'Risk [EUR]': [risk_at_minC,   risk_at_minR,   risk_at_minG],
    'Carbon [EUR]': [carbon_at_minC, carbon_at_minR, carbon_at_minG],
}, index=['min Cost', 'min Risk', 'min Carbon'])

print('Payoff Table:')
print(payoff.round(0))

cost_min, cost_max = payoff['Cost [EUR]'].min(),     payoff['Cost [EUR]'].max()
risk_min, risk_max = payoff['Risk [EUR]'].min(),     payoff['Risk [EUR]'].max()
carbon_min, carbon_max = payoff['Carbon [EUR]'].min(), payoff['Carbon [EUR]'].max()

print(f'\nRanges_')
print(f'Cost:   [{cost_min:,.0f}, {cost_max:,.0f}]   range={cost_max-cost_min:,.0f}')
print(f'Risk:   [{risk_min:,.0f}, {risk_max:,.0f}]                range={risk_max-risk_min:,.0f}')
print(f'Carbon: [{carbon_min:,.0f}, {carbon_max:,.0f}]         range={carbon_max-carbon_min:,.0f}')

Payoff Table:
            Cost [EUR]  Risk [EUR]  Carbon [EUR]
min Cost    24915676.0     10371.0      811573.0
min Risk    25764996.0         0.0      837659.0
min Carbon  26242891.0     13517.0      691803.0

Ranges_
Cost:   [24,915,676, 26,242,891]   range=1,327,215
Risk:   [0, 13,517]                range=13,517
Carbon: [691,803, 837,659]         range=145,856


In [78]:
eps = 1e-6
ranges = {
    'cost':   max(cost_max - cost_min,eps),
    'risk':   max(risk_max-risk_min, eps),
    'carbon': max(carbon_max-carbon_min,eps),
}

## 3.2. Weighted MOLP (4 company profiles)
Solve the aggregate weighted average MOLP: `min  w1 * CostNorm + w2 * RiskNorm + w3 * CarbonNorm` <br>
Since constants and dividing by a positive range don't change the argmin location for each term separately, the combined linear objective is: <br>
`c = w1 * c_cost / range_cost + w2 * c_risk / range_risk + w3 * c_carbon / range_carbon`

In [79]:
profiles = {
    'Cost-focused':  {'w1': 0.7, 'w2': 0.2, 'w3': 0.1},
    'Balanced':      {'w1': 0.34,'w2': 0.33,'w3': 0.33},
    'Risk-averse':   {'w1': 0.2, 'w2': 0.7, 'w3': 0.1},
    'Green-focused': {'w1': 0.2, 'w2': 0.1, 'w3': 0.7},
}

results = {}

for name, w in profiles.items():
    c_combined = (
        w['w1'] * c_cost   / ranges['cost'] +
        w['w2'] * c_risk   / ranges['risk'] +
        w['w3'] * c_carbon / ranges['carbon']
    )
    x_opt = solve_lp(c_combined)
    cost_v, risk_v, carbon_v = evaluate_objectives(x_opt)

    x1_opt = x_opt[:T]
    x2_opt = x_opt[T:]
    x3_opt = D - x1_opt - x2_opt

    results[name] = {
        'weights': w,
        'x1': x1_opt, 'x2': x2_opt, 'x3': x3_opt,
        'cost': cost_v, 'risk': risk_v, 'carbon': carbon_v,
        'cost_norm':   (cost_v   - cost_min)   / ranges['cost'],
        'risk_norm':   (risk_v   - risk_min)   / ranges['risk'],
        'carbon_norm': (carbon_v - carbon_min) / ranges['carbon'],
    }

    print(f'{name:15s} | Cost={cost_v:>12,.0f} EUR | Risk={risk_v:>10,.0f} EUR | Carbon={carbon_v:>14,.0f} EUR')

Cost-focused    | Cost=  24,990,139 EUR | Risk=     1,911 EUR | Carbon=       802,309 EUR
Balanced        | Cost=  25,272,040 EUR | Risk=     1,133 EUR | Carbon=       746,723 EUR
Risk-averse     | Cost=  25,162,051 EUR | Risk=       122 EUR | Carbon=       785,730 EUR
Green-focused   | Cost=  25,630,325 EUR | Risk=     7,247 EUR | Carbon=       697,940 EUR


In [80]:
summary = pd.DataFrame({
    name: {
        'w1 (Cost)':   r['weights']['w1'],
        'w2 (Risk)':   r['weights']['w2'],
        'w3 (Carbon)': r['weights']['w3'],
        'Cost [EUR]':       r['cost'],
        'Risk [EUR]':       r['risk'],
        'Carbon [EUR]':   r['carbon'],
        'Avg x1 [MWh]':     r['x1'].mean(),
        'Avg x2 [MWh]':     r['x2'].mean(),
        'Avg x3 [MWh]':     r['x3'].mean(),
        'x1 % of demand':   r['x1'].sum() / D.sum() * 100,
        'x2 % of demand':   r['x2'].sum() / D.sum() * 100,
        'x3 % of demand':   r['x3'].sum() / D.sum() * 100,
    }
    for name, r in results.items()
}).T

print('Optimal Portfolio by Company Profile')
summary.round(2)

Optimal Portfolio by Company Profile


,w1 (Cost),w2 (Risk),w3 (Carbon),Cost [EUR],Risk [EUR],Carbon [EUR],Avg x1 [MWh],Avg x2 [MWh],Avg x3 [MWh],x1 % of demand,x2 % of demand,x3 % of demand
Cost-focused,0.70,0.20,0.10,24990139.23,1911.46,802308.76,5.18,2.26,49.64,9.07,3.96,86.97
Balanced,0.34,0.33,0.33,25272039.72,1133.14,746723.46,4.26,5.50,47.32,7.47,9.63,82.91
Risk-averse,0.20,0.70,0.10,25162050.53,121.80,785730.17,3.89,3.21,49.97,6.82,5.62,87.56
Green-focused,0.20,0.10,0.70,25630325.50,7246.59,697939.84,4.54,8.69,43.85,7.95,15.22,76.83


# SAVE RESULTS

In [81]:
results_df = df[['demand_mwh', 'dayahead_price', 'balancing_price', 'ci_kgco2_mwh', 'scenario']].copy()
for name, r in results.items():
    safe_name = name.lower().replace('-', '_').replace(' ', '_')
    results_df[f'x1_{safe_name}'] = r['x1']
    results_df[f'x2_{safe_name}'] = r['x2']
    results_df[f'x3_{safe_name}'] = r['x3']

results_df.to_csv(PROCESSED_DIR / 'optimization_results.csv')
payoff.to_csv(PROCESSED_DIR / 'payoff_table.csv')
summary.to_csv(PROCESSED_DIR / 'profile_summary.csv')

print('Saved:')
print(f'  {PROCESSED_DIR}/optimization_results.csv')
print(f'  {PROCESSED_DIR}/payoff_table.csv')
print(f'  {PROCESSED_DIR}/profile_summary.csv')

Saved:
  C:\Users\Admin\Desktop\Project\energy-procurement-optimization\data\processed/optimization_results.csv
  C:\Users\Admin\Desktop\Project\energy-procurement-optimization\data\processed/payoff_table.csv
  C:\Users\Admin\Desktop\Project\energy-procurement-optimization\data\processed/profile_summary.csv
